# Khmer ASR baseline

This notebook evaluates the existing Khmer XLS-R checkpoint before project-specific fine-tuning. It uses the validated `voice_1000` manifests stored in Google Drive.

In [16]:
import sys
print(sys.version)
!nvidia-smi

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Mon Sep 28 06:10:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A

In [17]:
%cd /content
!ls scripts
!ls khmer_transliteration

/content
__pycache__
asr_text.py  __init__.py  __pycache__


In [18]:
!pip install -q -r requirements-asr.txt

In [19]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
source = Path('/content/drive/MyDrive/khmer_asr/voice_1000')
destination = Path('/content/data/voice_1000')
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source, destination, dirs_exist_ok=True)
!ls -lh /content/data/voice_1000

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 948K
-rw------- 1 root root 3.1K Sep 16 09:35 manifest_summary.json
-rw------- 1 root root  609 Sep 16 09:43 manifest_validation.json
-rw------- 1 root root  97K Sep 16 09:35 test.csv
-rw------- 1 root root 767K Sep 16 09:35 train.csv
-rw------- 1 root root  72K Sep 16 09:35 validation.csv


## Smoke test

The first run evaluates only three test samples. Audio is fetched from Hugging Face through the manifest source indexes.

In [29]:
!python scripts/run_asr_baseline.py --manifest-dir /content/data/voice_1000 --split test --max-samples 3 --output /content/data/voice_1000/baseline_smoke_test.json

Resolving data files: 100% 4447/4447 [00:00<00:00, 16753.87it/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 15769.49it/s]
config.json: 100% 2.07k/2.07k [00:00<00:00, 6.52MB/s]
tokenizer_config.json: 100% 306/306 [00:00<00:00, 1.84MB/s]
vocab.json: 100% 795/795 [00:00<00:00, 3.94MB/s]
added_tokens.json: 100% 23.0/23.0 [00:00<00:00, 102kB/s]
special_tokens_map.json: 100% 1.27k/1.27k [00:00<00:00, 3.21MB/s]

model.safetensors: downloading bytes:  18% 226M/1.26G [00:01<00:03, 261MB/s, 17.0MB/s  ]  
model.safetensors: downloading bytes:  25% 313M/1.26G [00:02<00:04, 220MB/s, 27.4MB/s  ]
model.safetensors: downloading bytes:  40% 509M/1.26G [00:02<00:02, 336MB/s, 41.9MB/s  ] ]
model.safetensors: downloading bytes:  50% 633M/1.26G [00:03<00:02, 311MB/s, 53.5MB/s  ] ]
model.safetensors: downloading bytes:  54% 679M/1.26G [00:03<00:02, 279MB/s, 56.3MB/s  ] ]
model.safetensors: downloading bytes:  62% 788M/1.26G [00:03<00:01, 273MB/s, 64.4MB/s  ] ]
model.safetensors: downloading bytes:  

In [30]:
import json
from pathlib import Path

report = json.loads(Path('/content/data/voice_1000/baseline_smoke_test.json').read_text(encoding='utf-8'))
print('Model:', report['model'])
print('Successful samples:', report['successful_samples'])
print('Metrics:', report['metrics'])
for item in report['predictions']:
    print('---')
    print('Reference:', item['reference'])
    print('Prediction:', item['prediction'])
    print('Latency:', item['latency_seconds'])

Model: vitouphy/wav2vec2-xls-r-300m-khmer
Successful samples: 3
Metrics: {'cer': 0.38333333333333336, 'wer': 3.56, 'sentence_accuracy': 0.0, 'mean_latency_seconds': 0.4963116583336766, 'median_latency_seconds': 0.28367933300069126, 'p95_latency_seconds': 1.1033050480000384, 'mean_real_time_factor': 0.04260843677980424}
---
Reference: យោងតាមហេតុការណ៍ ស្ថានភាពភូមិសាស្ត្រ និងលក្ខណៈនៃសម្បត្តិប្រាសាទ កោះកេរ បុរាណាចារ្យខ្មែរបានប្រឌិតរឿងព្រេងនិទានមួយគឺរឿង ស្តេចដំរីស។
Prediction: យើង តាម ហែតការ ស្ថាភាព ភូមិសាស្ត្រ និង លក្ខិណកនៃៃ សម្បត្ទ ប្រាសាទ កោះ គេ បុរាណាចា ខ្មែរ បាន ប្រដិត រឿង ប្រេង និទានមួយ គឺ រឿង ស្ដេច ដម្រី សរ
Latency: 1.1033050480000384
---
Reference: ហើយយើងនិយមហៅថា "ប្រាសាទកោះកេរ" គឺជាកោះនៃកិត្តិយស ឬកេរ្តិ៍ឈ្មោះ ។
Prediction: ហើយ យើង និយម ហៅៅ ថា ប្រាសាទ កះ គេ ឺ ជា កោះ នៃៃ កិត្តិយស ឺ គេ ឈ្មោ់
Latency: 0.10195059400030004
---
Reference: ប្រាសាទនេះមានកំពស់រហូតដល់ ៣៥ម៉ែត្រ ជាប្រាសាទមួយខ្ពស់ បន្ទាប់ពីប្រាសាទអង្គរវត្ត ប្រាសាទធំ ឬប្រាសាទប្រាង្គជាប្រាសាទភ្នំ និងជាទំរទ្រលិង្គ ទេវរាជដ៏ធំមួយដែលម

## Full test evaluation

Run this only after the smoke test successfully produces Khmer predictions. `--max-samples 0` evaluates the complete test split.

In [31]:
!python scripts/run_asr_baseline.py --manifest-dir /content/data/voice_1000 --split test --max-samples 0 --output /content/data/voice_1000/baseline_test.json

Resolving data files: 100% 4447/4447 [00:00<00:00, 26981.11it/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 26164.68it/s]
Loading weights: 100% 424/424 [00:00<00:00, 4824.94it/s]
Model: vitouphy/wav2vec2-xls-r-300m-khmer
Device: cuda
Successful samples: 126
Errors: 0
Resampled to 16 kHz: 0
Metrics: {'cer': 0.3489036299981574, 'wer': 5.932530120481927, 'sentence_accuracy': 0.0, 'mean_latency_seconds': 0.09304593160318507, 'median_latency_seconds': 0.08063255249999202, 'p95_latency_seconds': 0.13854440499926568, 'mean_real_time_factor': 0.013589264997638}
Saved report to: /content/data/voice_1000/baseline_test.json


In [32]:
from pathlib import Path
import shutil

source = Path('/content/data/voice_1000/baseline_test.json')
destination = Path('/content/drive/MyDrive/khmer_asr/voice_1000/baseline_test.json')
shutil.copy2(source, destination)
print('Saved baseline report to', destination)

Saved baseline report to /content/drive/MyDrive/khmer_asr/voice_1000/baseline_test.json


## Inspect baseline predictions

Review a few reference and prediction pairs before starting fine-tuning.

In [34]:
import json
from pathlib import Path

report = json.loads(Path('/content/drive/MyDrive/khmer_asr/voice_1000/baseline_test.json').read_text(encoding='utf-8'))
print('Model:', report['model'])
print('Decoder:', report['decoder'])
print('Successful samples:', report['successful_samples'])
print('Errors:', report['error_count'])
print('Metrics:', report['metrics'])

for index, item in enumerate(report['predictions'][:10], start=1):
    print(f'--- Sample {index} ---')
    print('Reference :', item['reference'])
    print('Prediction:', item['prediction'])
    print('Latency   :', round(item['latency_seconds'], 4), 'seconds')

Model: vitouphy/wav2vec2-xls-r-300m-khmer
Decoder: greedy_ctc
Successful samples: 126
Errors: 0
Metrics: {'cer': 0.3489036299981574, 'wer': 5.932530120481927, 'sentence_accuracy': 0.0, 'mean_latency_seconds': 0.09304593160318507, 'median_latency_seconds': 0.08063255249999202, 'p95_latency_seconds': 0.13854440499926568, 'mean_real_time_factor': 0.013589264997638}
--- Sample 1 ---
Reference : ដោយយោងទៅតាមរឿងនិទាន និងសៀវភៅសិលាចារឹកបុរាណបានចង្អុលបង្ហាញថា ស្ដេចឥណ្ឌាមួយអង្គមានព្រះនាមថា កៅណ្ឌិន្យ បានធ្វើដំណើរមកដល់ដែនកម្ពុជាបុរាណ។
Prediction: ដោយ យើង ទ តាម រឿង នាទៀន និង សៀវភៅ សីលា ចរឹក បុរាណ បាន ចង្អុល បង្ហាញ ថា ស្ដេច អន្ឌា មួយ អង្គ មាន ព្រះ នាម ថា កៅ ដិនបាន ធ្វើ ដំណើរ មក ដល់ អដែនកម្ពុជា បុរាន
Latency   : 0.5667 seconds
--- Sample 2 ---
Reference : សព្វថ្ងៃនេះ អ្នកស្រុកនៅទីជនបទនៅតែបន្តប្រើប្រាស់ផ្គាក់នៅក្នុងជីវភាពប្រចាំថ្ងៃ។
Prediction: សព្វយ ថ្ងៃ នេះ អ្នក ស្រុក នៅ ទី ជនបទ នៅ តែ បន្ត ប្រើ ប្រាស់ ផ្គក់ នៅ ក្នុង ជីវភាគ ប្រចាំ ថ្ងៃ
Latency   : 0.0824 seconds
--- Sample 3 ---
Reference : ក្នុងរាជ្យ